In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Random Forest: 30-Day Diabetes Readmission Prediction

이 노트북은 `diabetic_data.csv`를 사용하여 당뇨 환자의 **30일 이내 재입원 여부**를 예측하는 Random Forest baseline 모델을 만듭니다.

- Target: `readmitted`
- Positive class: `readmitted == '<30'`이면 1
- Negative class: `readmitted == 'NO'` 또는 `readmitted == '>30'`이면 0
- 모델: `RandomForestClassifier`

이전 Logistic Regression 노트북과 공정하게 비교할 수 있도록 같은 target 정의, 같은 train/test split 기준, 같은 주요 cleaning 기준을 사용합니다.

## 1. Import Libraries

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score
)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

RANDOM_STATE = 42
DATA_PATH = '/content/drive/MyDrive/SKKU/M2_1/diabetic_data.csv'
RESULT_PATH = 'random_forest_results.csv'

sns.set_theme(style='whitegrid')

## 2. Load Dataset

In [ ]:
# Google Drive에 저장된 CSV 파일을 불러옵니다.
df = pd.read_csv(DATA_PATH)

print('Dataset shape:', df.shape)
display(df.head())
display(df.info())

## 3. Define Target Variable

In [ ]:
# 30일 이내 재입원만 positive class로 정의합니다.
# '<30'이면 1, 'NO' 또는 '>30'이면 0입니다.
df['target_30day_readmission'] = (df['readmitted'] == '<30').astype(int)

print('Original readmitted distribution')
display(df['readmitted'].value_counts(dropna=False))

print('\nBinary target distribution')
display(df['target_30day_readmission'].value_counts(normalize=True).rename('ratio'))

plt.figure(figsize=(5, 4))
sns.countplot(data=df, x='target_30day_readmission')
plt.title('30-Day Readmission Target Distribution')
plt.xlabel('0 = No 30-day readmission, 1 = <30 days')
plt.ylabel('Count')
plt.show()

## 4. Basic Data Cleaning

In [ ]:
# '?'는 결측값을 의미하므로 NaN으로 변환합니다.
data = df.replace('?', np.nan).copy()

# 식별자 변수와 결측 비율이 매우 높은 변수는 제거합니다.
# readmitted 원본 변수는 이미 binary target으로 변환했으므로 feature에서 제외합니다.
drop_columns = [
    'encounter_id',
    'patient_nbr',
    'weight',
    'payer_code',
    'medical_specialty',
    'readmitted'
]

data = data.drop(columns=drop_columns)

# age는 '[50-60)' 같은 구간 문자열이므로 구간 중앙값으로 변환합니다.
def age_to_midpoint(age_value):
    if pd.isna(age_value):
        return np.nan
    left, right = age_value.strip('[]()').split('-')
    return (int(left) + int(right)) / 2

# ICD 진단 코드는 세부 코드가 많아 one-hot feature가 지나치게 많아질 수 있습니다.
# 수업 프로젝트에서 설명하기 쉽도록 큰 진단군으로 묶습니다.
def diagnosis_group(code):
    if pd.isna(code):
        return 'Missing'
    try:
        value = float(code)
    except ValueError:
        return 'Other'

    if 390 <= value <= 459 or value == 785:
        return 'Circulatory'
    if 460 <= value <= 519 or value == 786:
        return 'Respiratory'
    if 520 <= value <= 579 or value == 787:
        return 'Digestive'
    if 250 <= value < 251:
        return 'Diabetes'
    if 800 <= value <= 999:
        return 'Injury'
    if 710 <= value <= 739:
        return 'Musculoskeletal'
    if 580 <= value <= 629 or value == 788:
        return 'Genitourinary'
    if 140 <= value <= 239:
        return 'Neoplasms'
    return 'Other'

data['age_midpoint'] = data['age'].apply(age_to_midpoint)
data = data.drop(columns=['age'])

for col in ['diag_1', 'diag_2', 'diag_3']:
    data[f'{col}_group'] = data[col].apply(diagnosis_group)
data = data.drop(columns=['diag_1', 'diag_2', 'diag_3'])

# admission/discharge/source ID는 숫자형 크기보다 코드 범주가 의미 있으므로 범주형으로 처리합니다.
id_like_categorical_columns = ['admission_type_id', 'discharge_disposition_id', 'admission_source_id']
for col in id_like_categorical_columns:
    data[col] = data[col].astype('object')

print('Cleaned data shape:', data.shape)
display(data.head())

missing_summary = data.isna().mean().sort_values(ascending=False).head(15)
display((missing_summary * 100).round(2).to_frame('missing_percent'))

## 5. Train/Test Split

In [ ]:
# Logistic Regression notebook과 같은 기준으로 split합니다.
# random_state=42, test_size=0.2, stratify=y를 사용하므로 같은 데이터와 같은 전처리 기준이면 같은 test set이 됩니다.
X = data.drop(columns=['target_30day_readmission'])
y = data['target_30day_readmission']

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=RANDOM_STATE,
    stratify=y
)

print('X_train:', X_train.shape)
print('X_test:', X_test.shape)
print('\nTrain target ratio')
display(y_train.value_counts(normalize=True).rename('ratio'))
print('\nTest target ratio')
display(y_test.value_counts(normalize=True).rename('ratio'))

## 6. Preprocessing

In [ ]:
# 데이터 누수를 막기 위해 train/test split 이후에 전처리기를 정의하고 fit합니다.
# Random Forest는 scaling이 필수는 아니므로 StandardScaler는 사용하지 않습니다.
numeric_features = X_train.select_dtypes(include=['int64', 'float64']).columns.tolist()
categorical_features = X_train.select_dtypes(include=['object']).columns.tolist()

numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median'))
])

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='constant', fill_value='Missing')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features),
        ('cat', categorical_transformer, categorical_features)
    ]
)

print('Numeric features:', numeric_features)
print('\nCategorical features:', categorical_features)

## 7. Model: Random Forest

In [ ]:
# 전처리와 모델을 하나의 Pipeline으로 묶습니다.
# fit은 train set에서만 수행되고, test set에는 train set에서 학습된 전처리만 적용됩니다.
rf_model = RandomForestClassifier(
    n_estimators=300,
    random_state=RANDOM_STATE,
    class_weight='balanced',
    n_jobs=-1
)

rf_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', rf_model)
])

rf_pipeline.fit(X_train, y_train)

rf_pred = rf_pipeline.predict(X_test)
rf_proba = rf_pipeline.predict_proba(X_test)[:, 1]

print('Random Forest training complete.')

In [ ]:
# 선택 사항: 모델이 너무 복잡하거나 과적합이 의심될 때 아래처럼 하이퍼파라미터를 간단히 조정할 수 있습니다.
# 수업 프로젝트의 baseline 비교가 목적이라면 위 기본 모델 결과를 먼저 사용하세요.

# tuned_rf_model = RandomForestClassifier(
#     n_estimators=300,
#     max_depth=12,
#     min_samples_split=20,
#     min_samples_leaf=10,
#     random_state=RANDOM_STATE,
#     class_weight='balanced',
#     n_jobs=-1
# )
#
# tuned_rf_pipeline = Pipeline(steps=[
#     ('preprocessor', preprocessor),
#     ('model', tuned_rf_model)
# ])
#
# tuned_rf_pipeline.fit(X_train, y_train)
# tuned_rf_pred = tuned_rf_pipeline.predict(X_test)
# tuned_rf_proba = tuned_rf_pipeline.predict_proba(X_test)[:, 1]

## 8. Model Evaluation

In [ ]:
def evaluate_model(model_name, y_true, y_pred, y_proba):
    """Random Forest의 주요 binary classification 평가 지표를 계산하고 출력합니다."""
    metrics = {
        'Model': model_name,
        'Accuracy': accuracy_score(y_true, y_pred),
        'Precision': precision_score(y_true, y_pred, zero_division=0),
        'Recall': recall_score(y_true, y_pred, zero_division=0),
        'F1-score': f1_score(y_true, y_pred, zero_division=0),
        'ROC-AUC': roc_auc_score(y_true, y_proba)
    }

    print('=' * 80)
    print(model_name)
    print('=' * 80)
    print('Accuracy :', round(metrics['Accuracy'], 4))
    print('Precision:', round(metrics['Precision'], 4))
    print('Recall   :', round(metrics['Recall'], 4))
    print('F1-score :', round(metrics['F1-score'], 4))
    print('ROC-AUC  :', round(metrics['ROC-AUC'], 4))

    print('\nConfusion Matrix')
    cm = confusion_matrix(y_true, y_pred)
    display(pd.DataFrame(
        cm,
        index=['Actual 0', 'Actual 1'],
        columns=['Predicted 0', 'Predicted 1']
    ))

    print('\nClassification Report')
    print(classification_report(
        y_true,
        y_pred,
        target_names=['No 30-day readmission', '30-day readmission'],
        zero_division=0
    ))

    return metrics

rf_metrics = evaluate_model(
    'Random Forest',
    y_test,
    rf_pred,
    rf_proba
)

results_df = pd.DataFrame([rf_metrics])
metric_columns = ['Accuracy', 'Precision', 'Recall', 'F1-score', 'ROC-AUC']
results_df[metric_columns] = results_df[metric_columns].round(4)

display(results_df)

## 9. Feature Importance

In [ ]:
# Pipeline 안에서 학습된 전처리기와 Random Forest 모델을 꺼냅니다.
fitted_preprocessor = rf_pipeline.named_steps['preprocessor']
fitted_rf_model = rf_pipeline.named_steps['model']

# One-Hot Encoding 이후의 feature name을 가져와 feature importance와 정확히 매칭합니다.
feature_names = fitted_preprocessor.get_feature_names_out()
feature_importances = fitted_rf_model.feature_importances_

importance_df = pd.DataFrame({
    'feature': feature_names,
    'importance': feature_importances
}).sort_values('importance', ascending=False)

top_15_importance = importance_df.head(15)

display(top_15_importance)

plt.figure(figsize=(9, 6))
sns.barplot(data=top_15_importance.sort_values('importance'), x='importance', y='feature')
plt.title('Top 15 Random Forest Feature Importances')
plt.xlabel('Feature importance')
plt.ylabel('Feature')
plt.tight_layout()
plt.show()

## 10. Save Results

In [ ]:
# Random Forest 성능 결과를 CSV 파일로 저장합니다.
results_df.to_csv(RESULT_PATH, index=False)
print(f'Saved model results to {RESULT_PATH}')

# 필요하면 feature importance도 별도 CSV로 저장할 수 있습니다.
importance_df.to_csv('random_forest_feature_importance.csv', index=False)
print('Saved feature importance to random_forest_feature_importance.csv')

## 11. Interpretation

### Interpretation Notes

- 의료 재입원 예측에서는 Accuracy만으로 모델을 판단하기 어렵습니다. 실제 30일 이내 재입원 환자를 놓치는 것이 중요하므로 **Recall**, **F1-score**, **ROC-AUC**를 함께 봐야 합니다.
- Random Forest는 여러 decision tree를 결합하므로 Logistic Regression보다 **비선형 관계**와 **변수 간 상호작용**을 더 잘 반영할 수 있습니다.
- Feature importance 상위 변수는 모델이 재입원 예측에 많이 사용한 변수입니다. 예를 들어 `number_inpatient`, `number_emergency`, `time_in_hospital`, `num_medications`, `number_diagnoses` 같은 변수가 높게 나온다면, 과거 입원/응급 방문 경험, 입원 기간, 복용 약물 수, 진단 복잡도가 30일 이내 재입원 위험과 관련될 수 있다고 해석할 수 있습니다.
- One-Hot Encoding된 진단군, 인슐린 처방, 약물 변경 여부 같은 변수가 중요하게 나타난다면 특정 임상 상태나 치료 변화가 재입원 위험을 구분하는 데 도움이 되었을 가능성이 있습니다.
- 이 노트북은 하이퍼파라미터 튜닝보다 Logistic Regression notebook과 같은 기준에서 비교 가능한 Random Forest baseline을 만드는 데 초점을 둡니다.